# NSA vs Full Attention — expérience (Colab)

Ce notebook **clone** le dépôt [NativeSparseAttention](https://github.com/gaspardbd/NativeSparseAttention), installe les dépendances, lance `train_nsa_vs_full.py`, et **sauvegarde les figures** dans le répertoire de travail (et optionnellement sur Google Drive).

## Stratégie : clone vs autre

| Approche | Quand l’utiliser |
|----------|------------------|
| **`git clone` HTTPS** | **Recommandé sur Colab** : pas de clé SSH, fonctionne pour un repo public. Pour un repo **privé**, utilise un [Personal Access Token](https://github.com/settings/tokens) dans l’URL ou les secrets Colab. |
| **`git clone` SSH** (`git@github.com:...`) | Nécessite d’ajouter une clé SSH dans le runtime Colab (plus fragile). |
| **Upload du zip** | OK pour un run ponctuel, mais pas de `git pull` pour mettre à jour. |
| **Monter Google Drive** | Utile pour **persister** les PNG/JSON entre sessions ; le clone reste dans `/content` (éphémère). |

**Recommandation** : clone HTTPS + sauvegarde des figures dans `./outputs/` puis copie optionnelle vers Drive.

In [ ]:
# --- Config ---
REPO_URL_HTTPS = "https://github.com/gaspardbd/NativeSparseAttention.git"
REPO_DIR = "NativeSparseAttention"
OUTPUT_DIR = "outputs"  # sous le repo cloné

import os
import shutil
from pathlib import Path

# Dossier de sortie (persisté si tu copies vers Drive plus bas)
ROOT = Path("/content") if Path("/content").exists() else Path.cwd()
WORK = ROOT / REPO_DIR
OUT = WORK / OUTPUT_DIR
OUT.mkdir(parents=True, exist_ok=True)
print("WORK =", WORK)
print("OUT  =", OUT)

In [ ]:
%%capture
!pip install -q datasets transformers matplotlib tqdm

In [ ]:
# Clone (HTTPS — recommandé sur Colab)
# Si le dossier existe déjà : git pull au lieu de recloner
import subprocess

if WORK.exists():
    print("Dépôt déjà présent → git pull")
    subprocess.run(["git", "-C", str(WORK), "pull"], check=False)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL_HTTPS, str(WORK)], check=True)

print("Branche / commit :")
subprocess.run(["git", "-C", str(WORK), "log", "-1", "--oneline"], check=False)

In [ ]:
### Repo privé ou SSH

- **Privé** : `git clone https://<TOKEN>@github.com/gaspardbd/NativeSparseAttention.git` (ne commite jamais le token) ou [Colab Secrets](https://colab.research.google.com/notebooks/secrets.ipynb) + `GITHUB_TOKEN`.
- **SSH** (`github.com:gaspardbd/...`) : clé SSH dans le runtime Colab ; moins pratique que HTTPS.

In [ ]:
import os
import subprocess

# Script d'entraînement (à la racine du repo cloné)
script = WORK / "train_nsa_vs_full.py"
assert script.is_file(), (
    f"Introuvable : {script}\n"
    "Pousse `train_nsa_vs_full.py` sur GitHub ou mets à jour le clone (cellule git)."
)

os.chdir(WORK)
# Figures et JSON directement dans outputs/ (voir train_nsa_vs_full.py)
os.environ["NSA_OUTPUT_DIR"] = str(OUT)
print("CWD =", os.getcwd())
print("NSA_OUTPUT_DIR =", OUT)

# GPU recommandé : Runtime → Change runtime type → T4
subprocess.run([os.sys.executable, str(script)], check=True)

for name in ("nsa_vs_full.png", "results.json"):
    p = OUT / name
    print("OK" if p.is_file() else "manquant", "→", p)

In [ ]:
from IPython.display import Image, display

fig_path = OUT / "nsa_vs_full.png"
if fig_path.is_file():
    display(Image(filename=str(fig_path)))
else:
    print("Figure absente :", fig_path)

In [ ]:
# Optionnel : copier vers Google Drive (persistant entre sessions)
import shutil
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
    drive_out = Path("/content/drive/MyDrive/NSA_experiments")
    drive_out.mkdir(parents=True, exist_ok=True)
    for f in OUT.glob("*"):
        shutil.copy2(f, drive_out / f.name)
    print("Copié vers", drive_out)
except ImportError:
    print("Pas sur Colab → ignore le montage Drive.")